Stage 1 — Dataset quality: 
Class distribution, empty labels, annotation validation, duplicate detection.

In [ ]:
from pathlib import Path

# Dataset path relative to the notebooks directory
DATASET_PATH = Path("../data/RDD_SPLIT").resolve()

# Verify dataset exists
assert DATASET_PATH.is_dir(), f"Dataset not found: {DATASET_PATH}"

print(f"Dataset path: {DATASET_PATH}")

In [ ]:

import pandas as pd

dataset_stats = []

for split in ["train", "val", "test"]:
    images_dir = DATASET_PATH / split / "images"
    labels_dir = DATASET_PATH / split / "labels"

    image_files = [
        p for p in images_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    label_files = list(labels_dir.glob("*.txt"))

    dataset_stats.append({
        "Split": split,
        "Images": len(image_files),
        "Label Files": len(label_files),
        "Images Without Label Files": len(
            {p.stem for p in image_files}
            - {p.stem for p in label_files}
        )
    })

df_stats = pd.DataFrame(dataset_stats)
display(df_stats)


In [ ]:

from collections import Counter
import pandas as pd

class_counts = {}
empty_labels = {}

for split in ["train", "val", "test"]:
    labels_dir = DATASET_PATH / split / "labels"
    counts = Counter()
    empty_count = 0

    for label_path in labels_dir.glob("*.txt"):
        lines = label_path.read_text(encoding="utf-8").splitlines()

        if not lines:
            empty_count += 1

        for line in lines:
            parts = line.strip().split()
            if parts:
                class_id = int(parts[0])
                counts[class_id] += 1

    class_counts[split] = counts
    empty_labels[split] = empty_count

all_classes = sorted(
    set().union(*(counts.keys() for counts in class_counts.values()))
)

df_classes = pd.DataFrame({
    split: [class_counts[split][cls] for cls in all_classes]
    for split in ["train", "val", "test"]
}, index=all_classes)

df_classes.index.name = "Class ID"
df_classes["Total"] = df_classes.sum(axis=1)

display(df_classes)

print("\nEmpty annotation files:")
print(empty_labels)


## RDD2022 class-name mapping

The official RDD2022 documentation defines four core damage categories: D00 longitudinal crack, D10 transverse crack, D20 alligator crack, and D40 pothole. This local YOLO split contains five numeric classes, so class 4 is documented as the conversion-specific `Other corruption` category used by common five-class YOLO conversions. It should not be treated as one of the four core RDD2022 evaluation classes without checking the exact conversion provenance.

Sources: [RDD2022 paper](https://arxiv.org/abs/2209.08538), [official RoadDamageDetector repository](https://github.com/sekilab/RoadDamageDetector), and [CRDDC 2022 dataset record](https://figshare.com/articles/dataset/RDD2022_-_The_multi-national_Road_Damage_Dataset_released_through_CRDDC_2022/21431547).


In [ ]:
CLASS_NAMES = {
    0: "Longitudinal crack (D00)",
    1: "Transverse crack (D10)",
    2: "Alligator crack (D20)",
    3: "Pothole (D40)",
    4: "Other corruption (conversion-specific)",
}

class_name_table = pd.DataFrame(
    [{"Class ID": class_id, "Class Name": name} for class_id, name in CLASS_NAMES.items()]
).set_index("Class ID")

display(class_name_table)

# Use descriptive names in the class-distribution table while retaining numeric IDs.
df_classes_named = df_classes.rename(index=CLASS_NAMES)
display(df_classes_named)


In [ ]:

import matplotlib.pyplot as plt

ax = df_classes[["train", "val", "test"]].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Road Damage Class Distribution")
plt.xlabel("Class ID")
plt.ylabel("Number of Damage Instances")
plt.xticks(rotation=0)
plt.legend(title="Dataset Split")
plt.tight_layout()
plt.show()


In [ ]:
import hashlib
from collections import defaultdict

hash_to_files = defaultdict(list)
stem_to_splits = defaultdict(list)

for split in ["train", "val", "test"]:
    images_dir = DATASET_PATH / split / "images"
    for img_path in sorted(images_dir.iterdir()):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        stem_to_splits[img_path.stem].append(split)
        file_hash = hashlib.md5(img_path.read_bytes()).hexdigest()
        hash_to_files[file_hash].append((split, img_path.name))

overlapping_stems = {stem: splits for stem, splits in stem_to_splits.items() if len(splits) > 1}
duplicate_hashes = {h: files for h, files in hash_to_files.items() if len(files) > 1}
cross_split_duplicates = {
    h: files for h, files in duplicate_hashes.items()
    if len({split for split, _ in files}) > 1
}

print(f"Overlapping image filenames across splits: {len(overlapping_stems):,}")
print(f"Exact duplicate image groups (MD5): {len(duplicate_hashes):,}")
print(f"Cross-split exact duplicate groups: {len(cross_split_duplicates):,}")

if duplicate_hashes:
    df_duplicates = pd.DataFrame([
        {
            "MD5": h,
            "Count": len(files),
            "Splits": ", ".join(sorted({s for s, _ in files})),
            "Files": ", ".join(f"{s}/{name}" for s, name in files),
        }
        for h, files in duplicate_hashes.items()
    ])
    display(df_duplicates.head(20))


Stage 2 — Visual characteristics: 
Image resolutions, bounding-box statistics, visual samples and image quality.

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
SPLITS = ["train", "val", "test"]

def image_paths(split):
    return sorted(
        path for path in (DATASET_PATH / split / "images").iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    )

def load_image(path):
    image = mpimg.imread(path)
    if image.ndim == 2:
        image = np.repeat(image[..., None], 3, axis=2)
    return image[..., :3] if image.shape[-1] == 4 else image

print("Stage 2 helpers loaded.")


In [ ]:
resolution_rows = []
for split in SPLITS:
    for path in image_paths(split):
        image = load_image(path)
        height, width = image.shape[:2]
        resolution_rows.append({
            "Split": split,
            "Width": width,
            "Height": height,
            "Aspect Ratio": width / height,
            "Pixels (M)": width * height / 1_000_000,
        })

df_resolutions = pd.DataFrame(resolution_rows)
display(df_resolutions.groupby("Split")[["Width", "Height", "Aspect Ratio", "Pixels (M)"]].agg(["min", "median", "max"]).round(3))

resolution_counts = (
    df_resolutions.groupby(["Split", "Width", "Height"])
    .size()
    .reset_index(name="Images")
    .sort_values(["Split", "Images"], ascending=[True, False])
)
display(resolution_counts.groupby("Split").head(10))


In [ ]:
bbox_rows = []
invalid_annotations = []
for split in SPLITS:
    for label_path in sorted((DATASET_PATH / split / "labels").glob("*.txt")):
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
            parts = line.strip().split()
            if not parts:
                continue
            try:
                class_id, x_center, y_center, box_width, box_height = map(float, parts[:5])
            except (ValueError, IndexError):
                invalid_annotations.append({
                    "Split": split,
                    "Label File": label_path.name,
                    "Line": line_number,
                    "Reason": "Parse error / missing fields",
                    "Raw Content": line,
                })
                continue
            if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < box_width <= 1 and 0 < box_height <= 1):
                invalid_annotations.append({
                    "Split": split,
                    "Label File": label_path.name,
                    "Line": line_number,
                    "Reason": "Out-of-bounds or zero-dimension bbox",
                    "Raw Content": line,
                })
                continue
            bbox_rows.append({
                "Split": split,
                "Class ID": int(class_id),
                "Width": box_width,
                "Height": box_height,
                "Area": box_width * box_height,
                "Image": label_path.stem,
            })

df_bboxes = pd.DataFrame(bbox_rows)
display(df_bboxes.groupby(["Split", "Class ID"])[["Width", "Height", "Area"]].agg(["count", "mean", "median", "min", "max"]).round(4))
print(f"Valid bounding boxes: {len(df_bboxes):,}")
print(f"Invalid bounding-box rows: {len(invalid_annotations):,}")

if invalid_annotations:
    df_invalid = pd.DataFrame(invalid_annotations)
    display(df_invalid)


In [ ]:
fig, axes = plt.subplots(len(SPLITS), 4, figsize=(16, 11))
for row, split in enumerate(SPLITS):
    paths = image_paths(split)
    sample_indices = np.linspace(0, len(paths) - 1, 4, dtype=int)
    for col, index in enumerate(sample_indices):
        axes[row, col].imshow(load_image(paths[index]))
        axes[row, col].set_title(f"{split}: {paths[index].stem}")
        axes[row, col].axis("off")
plt.suptitle("Representative images from each dataset split")
plt.tight_layout()
plt.show()


In [ ]:
quality_rows = []
for split in SPLITS:
    paths = image_paths(split)
    sample_count = min(500, len(paths))
    sample_indices = np.linspace(0, len(paths) - 1, sample_count, dtype=int)
    for index in sample_indices:
        image = load_image(paths[index]).astype(np.float32)
        if image.max() > 1:
            image /= 255.0
        gray = image.mean(axis=2)
        quality_rows.append({
            "Split": split,
            "Mean Brightness": gray.mean(),
            "Brightness Std": gray.std(),
            "Dark Pixel %": (gray < 0.05).mean() * 100,
            "Bright Pixel %": (gray > 0.95).mean() * 100,
            "Edge Strength": (np.abs(np.diff(gray, axis=0)).mean() + np.abs(np.diff(gray, axis=1)).mean()) / 2,
        })

df_quality = pd.DataFrame(quality_rows)
display(df_quality.groupby("Split").agg(["mean", "median", "min", "max"]).round(4))
print(f"Image-quality sample size: {len(df_quality):,} images (up to 500 per split)")


Stage 3 — Domain analysis: 
Country distributions, capture-source differences and split composition.

In [ ]:
import re

def filename_metadata(path):
    tokens = path.stem.split("_")
    country = "United States" if tokens[:2] == ["United", "States"] else tokens[0]
    remainder = tokens[2:] if country == "United States" else tokens[1:]
    source_token = remainder[0] if remainder else ""
    if source_token in {"Drone", "MotorBike"}:
        source = source_token
    elif source_token and not source_token.isdigit():
        source = source_token
    else:
        source = "Unspecified/sequence"
    return country, source

metadata_rows = []
for split in SPLITS:
    for path in image_paths(split):
        country, source = filename_metadata(path)
        metadata_rows.append({
            "Split": split,
            "Image": path.stem,
            "Country": country,
            "Capture Source": source,
        })

df_metadata = pd.DataFrame(metadata_rows)
print("Metadata is inferred from image filename conventions; it is not embedded sensor metadata.")
print("\nCountry counts by split:")
display(pd.crosstab(df_metadata["Country"], df_metadata["Split"], margins=True))


In [ ]:
country_split = pd.crosstab(df_metadata["Country"], df_metadata["Split"], normalize="columns") * 100
print("Country composition within each split (%):")
display(country_split.round(2))

source_split = pd.crosstab(df_metadata["Capture Source"], df_metadata["Split"], margins=True)
print("Capture-source counts by split:")
display(source_split)

source_share = pd.crosstab(df_metadata["Capture Source"], df_metadata["Split"], normalize="columns") * 100
print("Capture-source composition within each split (%):")
display(source_share.round(2))


In [ ]:
class_split_rows = []
for split in SPLITS:
    for label_path in sorted((DATASET_PATH / split / "labels").glob("*.txt")):
        country, source = filename_metadata(label_path)
        class_ids = []
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if parts:
                try:
                    class_ids.append(int(parts[0]))
                except ValueError:
                    pass
        class_split_rows.append({
            "Split": split,
            "Image": label_path.stem,
            "Country": country,
            "Capture Source": source,
            "Has Annotation": bool(class_ids),
            "Annotation Count": len(class_ids),
            "Classes": sorted(set(class_ids)),
        })

df_composition = pd.DataFrame(class_split_rows)
df_composition = df_composition.explode("Classes").dropna(subset=["Classes"])
df_composition["Classes"] = df_composition["Classes"].astype(int)
print("Annotated-image counts by country, split, and class:")
display(df_composition.groupby(["Country", "Split", "Classes"]).size().unstack("Classes", fill_value=0))

print("\nImages with at least one annotation by country and split:")
annotated = df_composition.drop_duplicates(["Country", "Split", "Image"])
display(annotated.groupby(["Country", "Split"]).size().unstack("Split", fill_value=0))
